# Documentación del Proceso de Ingesta de Datos – Capa Bronze  
## Arquitectura Medallion en Databricks

## 1. Introducción

El presente notebook implementa un proceso de **ingesta de datos** correspondiente a la **capa Bronze** dentro de una arquitectura **Medallion**, utilizando la plataforma **Databricks**, **Apache Spark Structured Streaming** y **Delta Lake**.

El objetivo principal del proceso es transformar datos crudos almacenados en formato CSV en tablas **Delta Lake**, garantizando:
- Persistencia confiable de los datos
- Reprocesamientos seguros
- Evitación de duplicados
- Separación entre datos de negocio () y estado técnico del proceso

---

## 2. Rol de la Capa Bronze en la Arquitectura Medallion

Dentro de la arquitectura Medallion, la capa Bronze cumple la función de:

- Ingerir datos provenientes de fuentes externas
- Mantener los datos con mínima transformación
- Almacenar los datos en un formato eficiente y transaccional (Delta Lake)
- Servir como base para procesos posteriores de limpieza, normalización y modelado (capas Silver y Gold)

---

## 3. Descripción General del Proceso

El flujo de ingesta se estructura en tres etapas principales:

1. Lectura batch inicial para la inferencia del esquema
2. Lectura en streaming basada en archivos
3. Escritura de los datos en tablas Delta con manejo de checkpoint

Cada una de estas etapas se describe a continuación.

---

### 3.1 Lectura Batch para Inferencia de Esquema

En la primera etapa se realiza una lectura en modo **batch** de los archivos CSV, cuyo único propósito es **inferir el esquema de los datos** (tipos de columnas).
Los archivos se encuentran almacenados en un **Volume de Unity Catalog**, bajo la siguiente estructura:

/Volumes/pysparkdbt/source/source_data/{entity}/
Donde {entity} representa una carpeta asociada a cada entidad del dominio (por ejemplo: customers, trips, payments, etc.).

Apache Spark lee automáticamente todos los archivos CSV contenidos en cada carpeta, sin necesidad de especificar el nombre de los archivos individuales.

Es importante destacar que:
- En esta etapa no se persisten datos (no se guardan ni crean cosas)
- La lectura se utiliza exclusivamente para obtener el esquema
- El uso de inferSchema = true permite detectar los tipos de datos de manera automática

### 3.2 Lectura en Streaming Basada en Archivos
La segunda etapa implementa una lectura utilizando Structured Streaming, reutilizando el esquema previamente inferido.

Este tipo de streaming:
- No corresponde a una transmisión en tiempo real  
- Se basa en la detección de nuevos archivos en un directorio  
- Permite procesar datos de manera incremental  

El uso de streaming resulta necesario debido a que:  
- Structured Streaming requiere un esquema explícito  
- Habilita el uso de mecanismos de control de estado (checkpoint)  
- Evita el reprocesamiento de archivos ya consumidos  

Este patrón es ampliamente utilizado en Databricks para ingestas incrementales de datos basadas en archivos.  

### 3.3 Escritura en Delta Lake y Gestión del Checkpoint
Los datos procesados se escriben en formato Delta Lake dentro del catálogo correspondiente:

Catalog → pysparkdbt → bronze → Tables → {entity}
Estas tablas representan la capa Bronze del sistema y contienen los datos crudos convertidos a formato Delta, listos para ser consumidos por procesos posteriores.

3.3.1. Checkpoint del Proceso de Streaming
El estado del proceso de streaming se almacena en la siguiente ruta:
/Volumes/pysparkdbt/bronze/checkpoint/{entity}

Este checkpoint:
- No corresponde a una tabla
- No contiene datos de negocio

Almacena información técnica necesaria para el funcionamiento del streaming, tales como:  
- Offsets  
- Commits  
- Metadatos  
- Información sobre las fuentes procesadas 

Su función principal es permitir:  
- La reanudación del proceso ante fallos  
- La identificación de archivos ya procesados  
- La garantía de que cada archivo sea ingerido una única vez  
En Unity Catalog, estos elementos se visualizan dentro de la sección Volumes, y no dentro de Tables.

3.3.2. Uso del Trigger once  

El proceso utiliza el disparador trigger(once = true), el cual indica que:  
- Se procesan todos los archivos disponibles al momento de la ejecución  
- Se escriben los datos en las tablas Delta  
- Se guarda el estado del proceso en el checkpoint  
- La ejecución finaliza automáticamente    
En ejecuciones posteriores, únicamente se procesarán los archivos nuevos que no hayan sido previamente registrados en el checkpoint.  

---

### 4. Síntesis Conceptual del Proceso
1. Lectura Batch para Inferencia de Esquema    
Se realiza una lectura en modo batch de los archivos CSV únicamente para inferir el esquema de los datos, el cual será reutilizado en el proceso de streaming.  

2. Lectura en Streaming Basada en Archivos  
Se implementa una lectura mediante Structured Streaming para procesar de forma incremental los archivos CSV, identificando y consumiendo únicamente aquellos que no hayan sido procesados previamente.  

3. Escritura en Delta Lake y Gestión del Checkpoint  
Los datos ingeridos se persisten en tablas Delta correspondientes a la capa Bronze, mientras que el estado técnico del proceso se almacena en un checkpoint para garantizar idempotencia y reejecuciones seguras.  

In [0]:
entities = ['customers','trips','locations','payments','vehicles','drivers']

In [0]:
for entity in entities:

    # ============================================================
    # 🔹 1. Lectura batch (lote) inicial para obtener el esquema (schema), es decir los tipos de datos de cada columna
    # ============================================================
    df_batch = (
        spark.read.format("csv")                 # Indicamos que el archivo es CSV
            .option("header", "true")            # El archivo tiene encabezado
            .option("inferSchema", "true")       # Spark infiere automáticamente los tipos de datos
            .load(f"/Volumes/pysparkdbt/source/source_data/{entity}/")  # Ruta de los archivos CSV
    )

    # Guardamos el schema detectado para usarlo en el stream
    schema_entity = df_batch.schema


    # ============================================================
    # 🔹 2. Lectura en Streaming usando el mismo schema, osea para poner los datos en formato Dalta Lake y asi poder guardadolos en este formato
    # ============================================================
    df = (
        spark.readStream.format("csv")           # Lectura en streaming de archivos CSV
            .option("header", "true")            # El archivo tiene encabezado
            .schema(schema_entity)               # Reutilizamos el schema del batch para estabilidad
            .load(f"/Volumes/pysparkdbt/source/source_data/{entity}/")  # Misma ruta que en batch
    )


    # ============================================================
    # 🔹 3. Escritura en Delta Lake usando Structured Streaming
    # ============================================================
    (
        df.writeStream
            .format("delta")                     # Usamos formato Delta Lake como destino
            .outputMode("append")                # Agrega nuevos registros sin borrar los existentes
            .option(
                "checkpointLocation",            # Ruta donde Spark guarda offsets y estados
                f"/Volumes/pysparkdbt/bronze/checkpoint/{entity}"
            )
            .trigger(once=True)                  # Trigger once: procesa todo lo disponible y se detiene
            .toTable(f"pysparkdbt.bronze.{entity}")  # Guarda los datos en una tabla Delta del catálogo
    )